In [9]:
#TODO: load data
#TODO: train val test split
#TODO: missing data
#TODO: transformations/skewness
#TODO: correlation
#TODO: scaling

In [10]:
#imports
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.stats import skew

### Load data

In [ ]:
df = pd.read_csv("../../data/mock/new/data.csv")

# Clean column names, but keep all rows even if values repeat
df.columns = df.columns.str.strip()
df = df.drop(columns=[col for col in df.columns if col.lower().startswith("unnamed")], errors="ignore")
df = df.reset_index(drop=True)

print(f"Shape: {df.shape}")

display(df.head())
display(df.dtypes)
display(df.isna().sum().rename("missing_values"))


Shape: (2664, 21)


,max30minTemp,min30minTemp,mean30minTemp,trend30minTemp,std30minTemp,max30minHumidity,min30minHumidity,mean30minHumidity,trend30minHumidity,std30minHumidity,...,min30minLight,mean30minLight,trend30minLight,std30minLight,max30minCO2,min30minCO2,mean30minCO2,trend30minCO2,std30minCO2,rating
0,23.718,23.7,23.709000,0.000000,0.012728,26.29,26.272,26.28100,0.000000,0.012728,...,578.400000,581.800000,0.000000,4.808326,760.400000,749.2,754.800000,0.000000,7.919596,5
1,23.730,23.7,23.716000,0.010500,0.015100,26.29,26.230,26.26400,-0.025500,0.030790,...,572.666667,578.755556,-4.566667,6.274227,769.666667,749.2,759.755556,7.433333,10.248541,4
2,23.730,23.7,23.717625,0.005833,0.012750,26.29,26.125,26.22925,-0.046111,0.073907,...,493.750000,557.504167,-23.842593,42.810397,774.750000,749.2,763.504167,6.824074,11.235216,4
3,23.754,23.7,23.724900,0.009328,0.019661,26.29,26.125,26.22340,-0.029344,0.065328,...,488.600000,543.723333,-24.416146,48.208966,779.000000,749.2,766.603333,6.261979,11.945573,4
4,23.760,23.7,23.730750,0.010916,0.022684,26.29,26.125,26.22950,-0.018327,0.060312,...,488.600000,547.880556,-15.849340,44.305513,790.000000,749.2,770.502778,7.309736,14.331482,4


max30minTemp          float64
min30minTemp          float64
mean30minTemp         float64
trend30minTemp        float64
std30minTemp          float64
max30minHumidity      float64
min30minHumidity      float64
mean30minHumidity     float64
trend30minHumidity    float64
std30minHumidity      float64
max30minLight         float64
min30minLight         float64
mean30minLight        float64
trend30minLight       float64
std30minLight         float64
max30minCO2           float64
min30minCO2           float64
mean30minCO2          float64
trend30minCO2         float64
std30minCO2           float64
rating                  int64
dtype: object

max30minTemp          0
min30minTemp          0
mean30minTemp         0
trend30minTemp        0
std30minTemp          0
max30minHumidity      0
min30minHumidity      0
mean30minHumidity     0
trend30minHumidity    0
std30minHumidity      0
max30minLight         0
min30minLight         0
mean30minLight        0
trend30minLight       0
std30minLight         0
max30minCO2           0
min30minCO2           0
mean30minCO2          0
trend30minCO2         0
std30minCO2           0
rating                0
Name: missing_values, dtype: int64

### Train val test split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["rating"])
y = df["rating"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.1765,  # around 70% train, 15% validation, 15% test
    random_state=42,
    stratify=y_train_val
)

print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Validation: {X_val.shape}, {y_val.shape}")
print(f"Test: {X_test.shape}, {y_test.shape}")

display(pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "validation": y_val.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
}).fillna(0))


Feature columns:
['max30minTemp', 'min30minTemp', 'mean30minTemp', 'trend30minTemp', 'std30minTemp', 'max30minHumidity', 'min30minHumidity', 'mean30minHumidity', 'trend30minHumidity', 'std30minHumidity', 'max30minLight', 'min30minLight', 'mean30minLight', 'trend30minLight', 'std30minLight', 'max30minCO2', 'min30minCO2', 'mean30minCO2', 'trend30minCO2', 'std30minCO2']

Split sizes:
Train: (1864, 20), (1864,)
Validation: (400, 20), (400,)
Test: (400, 20), (400,)

Target distribution:


,train,validation,test
rating,,,
2,0.003219,0.0025,0.0025
3,0.531116,0.5325,0.5325
4,0.440987,0.4400,0.4400
5,0.024678,0.0250,0.0250


### Missing data

In [13]:
#For missing values we decided to keep the first version simple: if a row has a missing value, we remove that row. We expect the final dataset to have very few or no missing values, because the sensor/data preparation step should produce complete rows. Dropping rows is easier to explain and avoids guessing values with mean/median imputation. If we later see that many rows are removed, then we should reconsider and use imputation instead.
# TODO: change this

missing_before = df.isna().sum().sum()
rows_before = len(df)

df = df.dropna().reset_index(drop=True)

rows_after = len(df)
rows_removed = rows_before - rows_after

print(f"Missing values before: {missing_before}")
print(f"Rows before: {rows_before}")
print(f"Rows removed: {rows_removed}")
print(f"Rows after: {rows_after}")

display(df.isna().sum().rename("missing_values_after"))


Missing values before: 0
Rows before: 2664
Rows removed: 0
Rows after: 2664


max30minTemp          0
min30minTemp          0
mean30minTemp         0
trend30minTemp        0
std30minTemp          0
max30minHumidity      0
min30minHumidity      0
mean30minHumidity     0
trend30minHumidity    0
std30minHumidity      0
max30minLight         0
min30minLight         0
mean30minLight        0
trend30minLight       0
std30minLight         0
max30minCO2           0
min30minCO2           0
mean30minCO2          0
trend30minCO2         0
std30minCO2           0
rating                0
Name: missing_values_after, dtype: int64

### Outliers

In [14]:
# tranformations/skewness
def handle_skewness(df, threshold=0.75):
    df_transformed = df.copy()
    
    # Select only numerical columns
    numeric_cols = df_transformed.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        skew_val = skew(df_transformed[col])
        
        # Highly Positive Skew (Right Tailed)
        if skew_val > threshold:
            # np.log1p handles log(0) by calculating log(1+x)
            df_transformed[col] = np.log1p(df_transformed[col])
            print(f"Applied Log Transform to '{col}' (Skew: {skew_val:.2f})")
            
        # Highly Negative Skew (Left Tailed)
        elif skew_val < -threshold:
            # Squaring often helps pull the left tail in
            df_transformed[col] = np.square(df_transformed[col])
            print(f"Applied Square Transform to '{col}' (Skew: {skew_val:.2f})")
            
    return df_transformed

# Apply to your features
X_train_val_skewed = handle_skewness(X_train_val)

Applied Log Transform to 'max30minTemp' (Skew: 0.76)
Applied Log Transform to 'min30minTemp' (Skew: 0.83)
Applied Log Transform to 'mean30minTemp' (Skew: 0.79)
Applied Log Transform to 'trend30minTemp' (Skew: 1.85)
Applied Log Transform to 'std30minTemp' (Skew: 2.70)
Applied Log Transform to 'std30minHumidity' (Skew: 1.83)
Applied Log Transform to 'max30minLight' (Skew: 1.56)
Applied Log Transform to 'std30minLight' (Skew: 3.36)
Applied Log Transform to 'min30minCO2' (Skew: 0.89)
Applied Log Transform to 'mean30minCO2' (Skew: 0.78)
Applied Log Transform to 'std30minCO2' (Skew: 1.17)


In [15]:
#correlation


def combine_highly_correlated(df, threshold=0.8):
    iteration = 1
    
    while True:
        # 1. Calculate absolute correlation matrix
        corr_matrix = df.corr().abs()
        
        # 2. Select the upper triangle (to avoid self-corr and duplicates)
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
        # 3. Find pairs that exceed the threshold
        # We stack them and sort to find the highest correlation pair first
        pairs = upper[upper > threshold].stack().sort_values(ascending=False)
        
        if pairs.empty:
            print("Optimization complete: No more features exceed the correlation threshold.")
            break
            
        # 4. Take the top pair
        (feat1, feat2), correlation_value = pairs.index[0], pairs.values[0]
        
        # 5. Apply the formula: feature1_feature2 = feature1 * feature2
        new_col_name = f"{feat1}_{feat2}"
        df[new_col_name] = df[feat1] * df[feat2]
        
        print(f"Iteration {iteration}: Combining '{feat1}' and '{feat2}' (Corr: {correlation_value:.4f})")
        
        # 6. Drop the original features
        df = df.drop(columns=[feat1, feat2])
        iteration += 1
        
    return df

X_train_val_processed = combine_highly_correlated(X_train_val_skewed.copy())


Iteration 1: Combining 'min30minCO2' and 'mean30minCO2' (Corr: 0.9977)
Iteration 2: Combining 'max30minHumidity' and 'mean30minHumidity' (Corr: 0.9967)
Iteration 3: Combining 'max30minTemp' and 'mean30minTemp' (Corr: 0.9964)
Iteration 4: Combining 'max30minCO2' and 'min30minCO2_mean30minCO2' (Corr: 0.9946)
Iteration 5: Combining 'min30minHumidity' and 'max30minHumidity_mean30minHumidity' (Corr: 0.9933)
Iteration 6: Combining 'min30minTemp' and 'max30minTemp_mean30minTemp' (Corr: 0.9919)
Iteration 7: Combining 'min30minLight' and 'mean30minLight' (Corr: 0.9655)
Iteration 8: Combining 'max30minCO2_min30minCO2_mean30minCO2' and 'min30minHumidity_max30minHumidity_mean30minHumidity' (Corr: 0.9162)
Iteration 9: Combining 'max30minLight' and 'std30minLight' (Corr: 0.8680)
Iteration 10: Combining 'trend30minHumidity' and 'trend30minCO2' (Corr: 0.8639)
Iteration 11: Combining 'std30minHumidity' and 'trend30minHumidity_trend30minCO2' (Corr: 0.8688)
Optimization complete: No more features exceed 

In [16]:
#scaling


scaler = StandardScaler()

# 2. Fit and transform the training data
X_train_val_scaled = scaler.fit_transform(X_train_val_processed)

# 3. Convert back to DataFrame to keep column names
X_train_val_final = pd.DataFrame(X_train_val_scaled, columns=X_train_val_processed.columns)